In [1]:
#This writes files into Colab's filesystem
%%writefile requirements.txt
fastapi
uvicorn[standard]
python-multipart
soundfile
faster-whisper
pyngrok==5.1.0


Overwriting requirements.txt


In [2]:
%%writefile app.py
# app.py - FastAPI service for Whisper (faster-whisper)
from fastapi import FastAPI, UploadFile, File, HTTPException
from faster_whisper import WhisperModel
import soundfile as sf
import time, os

app = FastAPI(title="Whisper Transcription API (Colab)")

# ======= CONFIGURE MODEL HERE =======
# Recommended: use Systran converted models for faster-whisper
# If Colab cannot load large-v3 due to limits, change to "Systran/faster-whisper-medium"
MODEL_NAME = "Systran/faster-whisper-large-v3"   # or "Systran/faster-whisper-medium"
DEVICE = "cpu"          # Colab CPU runtime -> "cpu". If you have GPU runtime and drivers, use "cuda".
COMPUTE_TYPE = "int8"   # use quantized compute type to reduce memory
# ====================================

print("Loading model:", MODEL_NAME)
model = WhisperModel(MODEL_NAME, device=DEVICE, compute_type=COMPUTE_TYPE)
print("Model loaded.")

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):
    if not file.filename:
        raise HTTPException(status_code=400, detail="No file uploaded")

    # Save uploaded file
    tmp_path = f"/tmp/{time.time_ns()}_{file.filename}"
    with open(tmp_path, "wb") as f:
        f.write(await file.read())

    try:
        start = time.perf_counter()
        segments, info = model.transcribe(tmp_path, beam_size=5)
        transcription = " ".join([seg.text for seg in segments]).strip()
        latency = time.perf_counter() - start
        return {
            "transcription": transcription,
            "language": getattr(info, "language", None),
            "latency_seconds": round(latency, 3)
        }
    finally:
        try:
            os.remove(tmp_path)
        except Exception:
            pass

if __name__ == "__main__":
    import uvicorn
    uvicorn.run("app:app", host="0.0.0.0", port=8000)


Overwriting app.py


In [3]:
!pip install pyngrok fastapi uvicorn
from pyngrok import ngrok

# Paste your auth token from ngrok dashboard
NGROK_AUTH_TOKEN = "34a1xnR6x7Abd9GlPrYyOrgiJ5v_PDUaMf8eTqo74pgfWfdW"  # 🔹 replace with your token

ngrok.set_auth_token(NGROK_AUTH_TOKEN)


In [4]:
!pip install --upgrade pyngrok
!wget https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xvzf ngrok-v3-stable-linux-amd64.tgz
!mv ngrok /usr/local/bin/ngrok
!ngrok version


--2025-10-26 00:34:51--  https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
Resolving bin.equinox.io (bin.equinox.io)... 13.248.244.96, 75.2.60.68, 99.83.220.108, ...
Connecting to bin.equinox.io (bin.equinox.io)|13.248.244.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9322550 (8.9M) [application/octet-stream]
Saving to: ‘ngrok-v3-stable-linux-amd64.tgz.3’

ngrok-v3-stable-lin 100%[===================>]   8.89M  17.2MB/s    in 0.5s    

2025-10-26 00:34:52 (17.2 MB/s) - ‘ngrok-v3-stable-linux-amd64.tgz.3’ saved [9322550/9322550]

ngrok
ngrok version 3.32.0


In [5]:
!ngrok config add-authtoken 34a1xnR6x7Abd9GlPrYyOrgiJ5v_PDUaMf8eTqo74pgfWfdW


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [6]:
!pip install --upgrade pip setuptools wheel
!pip install "faster-whisper==1.0.0" ffmpeg
!apt-get install -y libavdevice-dev libavfilter-dev libopus-dev


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopus-dev is already the newest version (1.3.1-0.1build2).
libavdevice-dev is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
libavfilter-dev is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [7]:
!pip install huggingface_hub
from huggingface_hub import snapshot_download

# Download the small model manually
model_dir = snapshot_download(repo_id="openai/whisper-small")
print("✅ Model downloaded to:", model_dir)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Model downloaded to: /root/.cache/huggingface/hub/models--openai--whisper-small/snapshots/973afd24965f72e36ca33b3055d56a652f456b4d


In [8]:
from fastapi import FastAPI, UploadFile, File
from faster_whisper import WhisperModel
import uvicorn, threading
from pyngrok import ngrok

app = FastAPI()

# Use the model name directly instead of the local path
model_name = "small" # Changed from using model_path
model = WhisperModel(model_name, device="cpu", compute_type="int8")

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):
    data = await file.read()
    with open("temp.wav","wb") as f: f.write(data)
    segs,_ = model.transcribe("temp.wav")
    return {"text":" ".join([s.text for s in segs])}

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server).start()
public_url = ngrok.connect(8000).public_url
print("✅ Public URL:", public_url)
print("🌍 Swagger UI:", public_url + "/docs")

INFO:     Started server process [14713]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ Public URL: https://stephen-palaeontological-overplentifully.ngrok-free.dev
🌍 Swagger UI: https://stephen-palaeontological-overplentifully.ngrok-free.dev/docs
